# Week 5 Exercise — Medical Doctor Assistant RAG

**A medical doctor's assistant**

A RAG assistant over clinical medical knowledge base. Answer questions with details from a comprehensive medicine handbook.

**LLM backends:** Ollama (local) or OpenAI.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

## Config

- **KNOWLEDGE_BASE**: folder of `.md` (or other) docs. Default points to week5's `knowledge-base`; replaced with medicine knowledge folder.
- **LLM_PROVIDER**: `"ollama"` | `"openai"`.

In [ ]:
NOTEBOOK_DIR = Path(".").resolve()

# Knowledge base
KNOWLEDGE_BASE = NOTEBOOK_DIR / "../../knowledge-base"
DB_NAME = str(NOTEBOOK_DIR / "doctor_vector_db")

LLM_PROVIDER = "ollama"  # "ollama" | "openai"
OLLAMA_MODEL = "llama3.2"
OPENAI_MODEL = "gpt-4.1-nano"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
RETRIEVAL_K = 6

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## Ingest: load docs, chunk, embed, store in Chroma

Uses HuggingFace embeddings (no API key needed). Run once to build the vector store.

In [ ]:
def load_documents():
    docs = []
    base = Path(KNOWLEDGE_BASE)
    if not base.exists():
        raise FileNotFoundError(f"Knowledge file/folder not found: {base}.")
    for folder in base.iterdir():
        if folder.is_dir():
            loader = DirectoryLoader(
                str(folder),
                glob="**/*.md",
                loader_cls=TextLoader,
                loader_kwargs={"encoding": "utf-8"},
            )
            for doc in loader.load():
                doc.metadata["doc_type"] = folder.name
                docs.append(doc)
    return docs

text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

documents = load_documents()
chunks = text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} documents, {len(chunks)} chunks.")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)
print(f"Vector store ready: {DB_NAME}")

## Connect to existing vector store and set up retriever + LLM

If you already ran ingest above, you can skip re-running it and just load the store and pick the LLM.

In [ ]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

if api_key:
    LLM_PROVIDER = "openai"
    llm = ChatOpenAI(
        model_name=OPENAI_MODEL,
        temperature=0,
    )
else:
    LLM_PROVIDER = "ollama"
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

print(f"Using LLM: {LLM_PROVIDER}")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a medical doctor assistant with comprehensive knowledge base of clinical medicine. Answer the user's question using ONLY the context provided.
If the context does not contain enough information, say so. Do not invent response.

Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])
    return response.content

## Give it a try!

In [ ]:
answer_question("How do you do chest examination?", [])

## Gradio 🙌

In [ ]:
examples = [
    "What are the symptoms of flu?",
    "How do I manage a patient with Pneumonia",
    "How do I manage Heart failure?",
    "When should I see a doctor for chest pain?",
    "What are common side effects of antibiotics?"
]

gr.ChatInterface(answer_question, description="⚠️ This tool provides general health information and is not a substitute for professional medical advice.", title="Doctor's assistant", examples=examples).launch()